In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time
from datetime import datetime

# Setup
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.raw_data")
LANDING_PATH = "/Volumes/workspace/default/raw_data/clickstream_landing"
dbutils.fs.mkdirs(LANDING_PATH)

CLICKSTREAM_SCHEMA = StructType([
    StructField("id", LongType(), True),
    StructField("event_id", StringType(), True),
    StructField("user_id", StringType(), True),
    StructField("session_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("event_type", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("unit_price", DoubleType(), True),
    StructField("event_timestamp", TimestampType(), True),
])

def generate_user_metadata(num_users=1000):
    """Generate user dimension with PII."""
    return (
        spark.range(1, num_users + 1)
        .withColumns({
            "user_id": F.concat(F.lit("USR_"), F.col("id")),
            "full_name": F.concat(F.lit("User_"), F.col("id")),
            "email": F.concat(F.lit("user_"), F.col("id"), F.lit("@company.com")),
            "credit_card_num": F.concat(
                F.lit("4532-"), (F.rand() * 8999 + 1000).cast("int"),
                F.lit("-"), (F.rand() * 8999 + 1000).cast("int"),
                F.lit("-"), (F.rand() * 8999 + 1000).cast("int"),
            ),
            "ip_address": F.concat(
                (F.rand() * 200 + 10).cast("int"), F.lit("."),
                (F.rand() * 250).cast("int"), F.lit(".1.100"),
            ),
        })
        .drop("id")
    )

def generate_streaming_batch(batch_id, records_per_batch=2000, inject_bad_records=True):
    """Generate clickstream events as JSON files."""
    df = (
        spark.range(0, records_per_batch)
        .withColumns({
            "event_id": F.expr("uuid()"),
            "user_id": F.concat(F.lit("USR_"), (F.rand() * 1000 + 1).cast("int")),
            "session_id": F.expr("uuid()"),
            "product_id": F.concat(F.lit("PROD_"), (F.rand() * 50 + 1).cast("int")),
            "event_type": F.element_at(
                F.array(F.lit("view"), F.lit("cart"), F.lit("purchase")),
                (F.rand() * 3 + 1).cast("int"),
            ),
            "quantity": (F.rand() * 5 + 1).cast("int"),
            "unit_price": F.round(F.rand() * 100 + 5, 2),
            "event_timestamp": F.current_timestamp(),
        })
    )

    # Inject bad records for testing
    if inject_bad_records and batch_id % 2 == 0:
        current_time = datetime.now()
        dirty_df = spark.createDataFrame([
            (9999, "", "USR_999", "sess_bad", "PROD_1", "purchase", -5, 10.0, current_time),
            (9998, "evt_corrupt", "USR_998", "sess_bad", "PROD_2", "view", 0, -50.0, current_time),
        ], CLICKSTREAM_SCHEMA)
        df = df.union(dirty_df)

    output_file = f"{LANDING_PATH}/batch_{batch_id}_{int(time.time())}.json"
    df.coalesce(1).write.mode("overwrite").json(output_file)
    return output_file

def generate_multiple_batches(start_batch_id, num_batches=5, records_per_batch=2000, inject_bad_records=True):
    """Generate multiple batches of clickstream data."""
    files = []
    for i in range(num_batches):
        batch_id = start_batch_id + i
        file_path = generate_streaming_batch(
            batch_id=batch_id,
            records_per_batch=records_per_batch,
            inject_bad_records=inject_bad_records,
        )
        files.append(file_path)
    return files

# Initial data generation
user_dim = generate_user_metadata(1000)
user_dim.write.format("delta").mode("overwrite").saveAsTable("workspace.default.dim_users")

generated_files = generate_multiple_batches(start_batch_id=1, num_batches=5)
print(f"✅ Created dim_users (1,000 users) and {len(generated_files)} batches")

✅ Created dim_users (1,000 users) and 5 batches


In [0]:
# List files
for f in dbutils.fs.ls(LANDING_PATH):
    print(f"  {'[DIR]' if f.isDir() else '[FILE]'} {f.name}")

# Read and display
df = spark.read.option("recursiveFileLookup", "true").json(LANDING_PATH)
print(f"\nTotal records: {df.count()}")
df.printSchema()

print("\nSample events:")
display(df.orderBy("event_timestamp").limit(20))

# Check for bad records
bad = df.filter((F.col("quantity") < 0) | (F.col("unit_price") < 0))
if bad.count() > 0:
    print(f"\nBad records: {bad.count()}")
    display(bad)
else:
    print("\nNo bad records")

  [DIR] batch_1_1788331802.json/
  [DIR] batch_2_1788331804.json/
  [DIR] batch_3_1788331805.json/
  [DIR] batch_4_1788331806.json/
  [DIR] batch_5_1788331807.json/

Total records: 10004
root
 |-- event_id: string (nullable = true)
 |-- event_timestamp: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- id: long (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: long (nullable = true)
 |-- session_id: string (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- user_id: string (nullable = true)


Sample events:


event_id,event_timestamp,event_type,id,product_id,quantity,session_id,unit_price,user_id
ce4a5b4a-bbd6-4ac8-a36f-eb26f57459e0,2026-09-02T06:50:03.187Z,cart,3,PROD_13,4,df02456a-6b15-4bea-90ac-db4ddd216de8,34.8,USR_945
a7765912-c947-4aea-b43d-df0aefa99f8d,2026-09-02T06:50:03.187Z,purchase,11,PROD_43,4,aca9e5e6-cef9-4e32-b08f-115f5b4b65de,102.36,USR_308
ad48be90-d281-49db-93f6-42ba2c059a98,2026-09-02T06:50:03.187Z,cart,0,PROD_40,4,727df450-e62c-4350-a018-886d04c1b9cd,93.52,USR_452
a11a19a7-1363-41c0-9d8f-8c32c8056526,2026-09-02T06:50:03.187Z,view,2,PROD_40,1,585efcc3-64c9-4c2a-b905-995666a5dd7e,5.94,USR_553
ab80c722-231b-4b45-bc12-a89accd5c482,2026-09-02T06:50:03.187Z,cart,7,PROD_15,1,a7efc733-edac-45d6-9b4b-e48a9e3d7dbd,86.65,USR_739
21c1d7da-c085-4b24-98f5-aec0df544c8f,2026-09-02T06:50:03.187Z,purchase,4,PROD_23,3,fc369869-1f16-4da8-9f6b-8f9250e36157,35.41,USR_568
c1f2ae4e-da8b-4209-938a-be665f518794,2026-09-02T06:50:03.187Z,view,15,PROD_32,3,738fb0e7-8510-44ec-be02-2e1e329054aa,14.73,USR_105
d5fa223d-fee3-4850-b764-781920114640,2026-09-02T06:50:03.187Z,view,17,PROD_11,2,8c4fb10d-dad8-491b-b0f2-7c9231be1f8e,11.8,USR_847
d406845a-cce1-42e5-9515-5c5e5051e793,2026-09-02T06:50:03.187Z,view,6,PROD_42,3,020ef9cf-3349-4b51-b5cb-e536cd9da4da,65.06,USR_164
434a6552-be70-4e20-b851-d7c4e12d920c,2026-09-02T06:50:03.187Z,view,12,PROD_15,3,bcd0e745-380a-4e2e-a8eb-777302b339c0,43.84,USR_255



Bad records: 4


event_id,event_timestamp,event_type,id,product_id,quantity,session_id,unit_price,user_id
,2026-09-02T06:50:06.506Z,purchase,9999,PROD_1,-5,sess_bad,10.0,USR_999
evt_corrupt,2026-09-02T06:50:06.506Z,view,9998,PROD_2,0,sess_bad,-50.0,USR_998
,2026-09-02T06:50:04.082Z,purchase,9999,PROD_1,-5,sess_bad,10.0,USR_999
evt_corrupt,2026-09-02T06:50:04.082Z,view,9998,PROD_2,0,sess_bad,-50.0,USR_998


In [0]:
from pyspark.sql import functions as F

SOURCE_PATH = "/Volumes/workspace/default/raw_data/clickstream_landing"
CHECKPOINT_PATH = "/Volumes/workspace/default/raw_data/checkpoints/clickstream_bronze"
TARGET_TABLE = "workspace.default.clickstream_bronze"

def process_new_data():
    """Process new files with Auto Loader (availableNow trigger)."""
    df_stream = (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", f"{CHECKPOINT_PATH}/schema")
        .option("cloudFiles.inferColumnTypes", "true")
        .option("cloudFiles.schemaHints", "event_timestamp TIMESTAMP")
        .load(SOURCE_PATH)
    )

    df_bronze = df_stream.select(
        "*",
        F.col("_metadata.file_path").alias("source_file"),
        F.current_timestamp().alias("ingestion_timestamp"),
    )

    # Track state before
    if spark.catalog.tableExists(TARGET_TABLE):
        count_before = spark.table(TARGET_TABLE).count()
        files_before = {r.source_file for r in spark.table(TARGET_TABLE).select("source_file").distinct().collect()}
    else:
        count_before = 0
        files_before = set()

    query = (
        df_bronze.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", CHECKPOINT_PATH)
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .toTable(TARGET_TABLE)
    )
    query.awaitTermination()

    # Track state after
    bronze = spark.table(TARGET_TABLE)
    count_after = bronze.count()
    files_after = {r.source_file for r in bronze.select("source_file").distinct().collect()}

    print(f"✅ Processed {count_after - count_before:,} new records from {len(files_after - files_before)} new files")
    print(f"   Total: {count_after:,} records, {len(files_after)} files")

# Run initial load
process_new_data()

✅ Processed 10,004 new records from 5 new files
   Total: 10,004 records, 5 files


In [0]:
from pyspark.sql import functions as F

# Enable Change Data Feed (CDF) on bronze table
spark.sql("""
    ALTER TABLE workspace.default.clickstream_bronze
    SET TBLPROPERTIES ('delta.enableChangeDataFeed' = true)
""")

def get_last_cdf_version(silver_table):
    """Retrieve the last processed CDF version stored as a table property on the silver table."""
    try:
        props = spark.sql(f"SHOW TBLPROPERTIES {silver_table} ('last_cdf_version')").collect()
        if props and props[0]["value"] and props[0]["value"].strip():
            return int(props[0]["value"])
    except:
        pass
    return None

def set_last_cdf_version(silver_table, version):
    """Store the last processed CDF version as a table property on the silver table."""
    spark.sql(f"ALTER TABLE {silver_table} SET TBLPROPERTIES ('last_cdf_version' = '{version}')")

def mask_pii_and_create_silver(bronze_table="workspace.default.clickstream_bronze",
                                 user_table="workspace.default.dim_users",
                                 silver_table="workspace.default.clickstream_silver_masked",
                                 mode="incremental"):
    """
    Masks PII data from user dimension and joins with bronze clickstream data.
    Uses Change Data Feed (CDF) for reliable incremental processing.
    """
    silver_exists = spark.catalog.tableExists(silver_table)
    
    # Determine which bronze records to process using Change Data Feed
    if mode == "incremental" and silver_exists:
        # Get the last processed CDF version from the silver table property
        last_version = get_last_cdf_version(silver_table)
        
        if last_version is None:
            print("⚠️  No CDF version tracked - switching to INITIAL mode")
            mode = "initial"
            bronze_df = spark.table(bronze_table)
        else:
            start_version = last_version + 1
            print(f"📊 Reading CDF changes from version {start_version} onwards")

            bronze_df = (
                spark.read.format("delta")
                .option("readChangeFeed", "true")
                .option("startingVersion", start_version)
                .table(bronze_table)
            )
            bronze_df = bronze_df.filter(F.col("_change_type") == "insert")
            bronze_df = bronze_df.drop("_change_type", "_commit_version", "_commit_timestamp")

            new_record_count = bronze_df.count()
            if new_record_count == 0:
                print("✅ No new CDF changes to process!")
                return {"status": "no_new_data", "records_processed": 0, "mode": mode}
            print(f"   Found {new_record_count:,} new records from CDF")
    else:
        # Initial mode or silver doesn't exist - process all bronze data
        if mode == "incremental" and not silver_exists:
            print("⚠️  Silver table doesn't exist - switching to INITIAL mode")
            mode = "initial"
        bronze_df = spark.table(bronze_table)
    
    # Apply PII masking
    users_masked = spark.table(user_table).select(
        "user_id",
        F.concat(F.substring(F.col("full_name"), 1, 2), F.lit("***")).alias("masked_name"),
        F.concat(F.lit("***@"), F.element_at(F.split(F.col("email"), "@"), 2)).alias("masked_email"),
        F.concat(F.lit("****-****-****-"), F.substring(F.col("credit_card_num"), -4, 4)).alias("masked_credit_card"),
        F.concat(
            F.element_at(F.split(F.col("ip_address"), "\\."), 1), F.lit("."),
            F.element_at(F.split(F.col("ip_address"), "\\."), 2), F.lit(".***.**"),
        ).alias("masked_ip"),
    )

    silver_df = bronze_df.join(users_masked, "user_id", "left")
    records_to_process = silver_df.count()
    
    # Write to silver table
    if mode == "initial":
        silver_df.write.format("delta").mode("overwrite").saveAsTable(silver_table)
        print(f"✅ Silver table created with {records_to_process:,} records")
    else:
        silver_df.write.format("delta").mode("append").saveAsTable(silver_table)
        print(f"✅ Added {records_to_process:,} new records to silver table")

    # Track the latest CDF version from bronze
    latest_version = (
        spark.sql(f"DESCRIBE HISTORY {bronze_table}")
        .select("version")
        .orderBy(F.col("version").desc())
        .first()["version"]
    )
    set_last_cdf_version(silver_table, latest_version)

    total_records = spark.table(silver_table).count()
    print(f"   Total: {total_records:,} records (CDF version: {latest_version})")

    display(spark.table(silver_table).orderBy(F.col("ingestion_timestamp").desc()).limit(10))

    return {
        "status": "success",
        "records_processed": records_to_process,
        "total_records": total_records,
        "mode": mode,
        "last_cdf_version": latest_version,
    }

# Run initial load
result = mask_pii_and_create_silver(mode="initial")

✅ Silver table created with 10,004 records
   Total: 10,004 records (CDF version: 2)


user_id,event_id,event_timestamp,event_type,id,product_id,quantity,session_id,unit_price,_rescued_data,source_file,ingestion_timestamp,masked_name,masked_email,masked_credit_card,masked_ip
USR_424,a11a2265-5b96-4c73-839d-27ae3c2463d2,2026-09-02T06:50:06.868Z,purchase,8,PROD_12,4,f4ffd974-10b2-4afc-939c-2286d9aecb65,96.05,null,/Volumes/workspace/default/raw_data/clickstream_landing/batch_4_1788331806.json/part-00000-tid-6663043186296468468-e79c0fc0-04c0-487a-ba70-69b7db1e739a-976-1-c000.json,2026-09-02T06:50:19.518Z,Us***,***@company.com,****-****-****-1777,173.218.***.**
USR_471,000609bc-c220-4c3b-ae9c-92b1d782c169,2026-09-02T06:50:06.868Z,view,5,PROD_34,2,5639789f-131c-48b5-b31d-e421107d2a91,101.33,null,/Volumes/workspace/default/raw_data/clickstream_landing/batch_4_1788331806.json/part-00000-tid-6663043186296468468-e79c0fc0-04c0-487a-ba70-69b7db1e739a-976-1-c000.json,2026-09-02T06:50:19.518Z,Us***,***@company.com,****-****-****-3692,39.41.***.**
USR_13,f5ce1df6-f0ce-4e4b-b1c5-7d0d65b01166,2026-09-02T06:50:06.868Z,purchase,6,PROD_16,2,13a7b475-7a78-437e-b107-80df25f77193,44.6,null,/Volumes/workspace/default/raw_data/clickstream_landing/batch_4_1788331806.json/part-00000-tid-6663043186296468468-e79c0fc0-04c0-487a-ba70-69b7db1e739a-976-1-c000.json,2026-09-02T06:50:19.518Z,Us***,***@company.com,****-****-****-4609,72.210.***.**
USR_734,07cc19c8-5112-4916-9dfb-05fca0a05eff,2026-09-02T06:50:06.868Z,view,7,PROD_32,4,d5497a13-f23d-4e58-b44f-23d97ab0ef5d,56.67,null,/Volumes/workspace/default/raw_data/clickstream_landing/batch_4_1788331806.json/part-00000-tid-6663043186296468468-e79c0fc0-04c0-487a-ba70-69b7db1e739a-976-1-c000.json,2026-09-02T06:50:19.518Z,Us***,***@company.com,****-****-****-5581,133.103.***.**
USR_914,8f505761-e6db-4294-8adf-5598d73a053f,2026-09-02T06:50:06.868Z,cart,1,PROD_49,2,bd1b27be-f1dc-4e8a-a086-a3aac1aadfd5,53.16,null,/Volumes/workspace/default/raw_data/clickstream_landing/batch_4_1788331806.json/part-00000-tid-6663043186296468468-e79c0fc0-04c0-487a-ba70-69b7db1e739a-976-1-c000.json,2026-09-02T06:50:19.518Z,Us***,***@company.com,****-****-****-4491,206.142.***.**
USR_672,e094d0d9-1540-4fc7-b4b9-5cc00df97e40,2026-09-02T06:50:06.868Z,purchase,4,PROD_49,1,ba32d11b-688c-4a56-aa7c-3add8ca04202,100.05,null,/Volumes/workspace/default/raw_data/clickstream_landing/batch_4_1788331806.json/part-00000-tid-6663043186296468468-e79c0fc0-04c0-487a-ba70-69b7db1e739a-976-1-c000.json,2026-09-02T06:50:19.518Z,Us***,***@company.com,****-****-****-9848,204.122.***.**
USR_375,65dcdf8a-821f-4a5f-8423-675970efe7d6,2026-09-02T06:50:06.868Z,purchase,0,PROD_33,5,9f96f8ed-52f1-42b3-a1c0-4c43e4abe31d,41.12,null,/Volumes/workspace/default/raw_data/clickstream_landing/batch_4_1788331806.json/part-00000-tid-6663043186296468468-e79c0fc0-04c0-487a-ba70-69b7db1e739a-976-1-c000.json,2026-09-02T06:50:19.518Z,Us***,***@company.com,****-****-****-8070,74.103.***.**
USR_85,b399b79a-47f4-4d88-9ec8-9e4b0f3f9ab1,2026-09-02T06:50:06.868Z,view,3,PROD_23,1,38b94eee-a7a8-4d34-b51b-4812587c2e66,95.45,null,/Volumes/workspace/default/raw_data/clickstream_landing/batch_4_1788331806.json/part-00000-tid-6663043186296468468-e79c0fc0-04c0-487a-ba70-69b7db1e739a-976-1-c000.json,2026-09-02T06:50:19.518Z,Us***,***@company.com,****-****-****-2888,123.17.***.**
USR_414,657e7134-ef84-40d5-8a5f-8828a2465197,2026-09-02T06:50:06.868Z,view,2,PROD_8,3,315391bd-f087-4eb0-a006-bc3d182be0eb,90.56,null,/Volumes/workspace/default/raw_data/clickstream_landing/batch_4_1788331806.json/part-00000-tid-6663043186296468468-e79c0fc0-04c0-487a-ba70-69b7db1e739a-976-1-c000.json,2026-09-02T06:50:19.518Z,Us***,***@company.com,****-****-****-5761,160.172.***.**
USR_724,86656759-d8d6-41a2-8349-f690519a54e7,2026-09-02T06:50:06.868Z,view,9,PROD_43,3,e006d6b6-48f2-40d7-816b-a6f7b05fe87a,90.42,null,/Volumes/workspace/default/raw_data/clickstream_landing/batch_4_1788331806.json/part-00000-tid-6663043186296468468-e79c0fc0-04c0-487a-ba70-69b7db1e739a-976-1-c000.json,2026-09-02T06:50:19.518Z,Us***,***

In [0]:
# Generate a new batch of clickstream data
generate_streaming_batch(batch_id=6, records_per_batch=1000)

'/Volumes/workspace/default/raw_data/clickstream_landing/batch_6_1788332355.json'

In [0]:
# Process new files with Auto Loader
process_new_data()

✅ Processed 1,002 new records from 1 new files
   Total: 11,006 records, 6 files


In [0]:
# Apply PII masking using CDC
mask_pii_and_create_silver()

📊 Reading CDF changes from version 3 onwards
   Found 1,002 new records from CDF
✅ Added 1,002 new records to silver table
   Total: 11,006 records (CDF version: 3)


user_id,event_id,event_timestamp,event_type,id,product_id,quantity,session_id,unit_price,_rescued_data,source_file,ingestion_timestamp,masked_name,masked_email,masked_credit_card,masked_ip
USR_637,d1b532c7-2580-4ce6-99ed-31b6d101f822,2026-09-02T06:59:15.946Z,cart,0,PROD_23,5,32bf88dd-c89c-4169-840a-da5d419c5366,25.44,null,/Volumes/workspace/default/raw_data/clickstream_landing/batch_6_1788332355.json/part-00000-tid-7492778453526864519-931ec837-d246-46dc-9ddd-bd773233d7de-1065-1-c000.json,2026-09-02T06:59:21.866Z,Us***,***@company.com,****-****-****-6527,198.163.***.**
USR_321,95dd6af8-559b-4b87-bea1-8eda297aa3e2,2026-09-02T06:59:15.946Z,cart,4,PROD_42,2,07d06f6c-eeea-460e-87f9-cf51425acd39,101.47,null,/Volumes/workspace/default/raw_data/clickstream_landing/batch_6_1788332355.json/part-00000-tid-7492778453526864519-931ec837-d246-46dc-9ddd-bd773233d7de-1065-1-c000.json,2026-09-02T06:59:21.866Z,Us***,***@company.com,****-****-****-8257,77.66.***.**
USR_180,651ec749-ad60-4cb0-8a0d-82975cdb920d,2026-09-02T06:59:15.946Z,purchase,6,PROD_16,5,2f6b83a0-c1d5-4a61-80f2-52bfe514c979,9.27,null,/Volumes/workspace/default/raw_data/clickstream_landing/batch_6_1788332355.json/part-00000-tid-7492778453526864519-931ec837-d246-46dc-9ddd-bd773233d7de-1065-1-c000.json,2026-09-02T06:59:21.866Z,Us***,***@company.com,****-****-****-6614,111.66.***.**
USR_82,e84df535-90b9-4783-9e1f-4c5f554d2145,2026-09-02T06:59:15.946Z,view,2,PROD_37,4,504f45d1-f436-4810-9e6e-f75da8f8ca62,91.8,null,/Volumes/workspace/default/raw_data/clickstream_landing/batch_6_1788332355.json/part-00000-tid-7492778453526864519-931ec837-d246-46dc-9ddd-bd773233d7de-1065-1-c000.json,2026-09-02T06:59:21.866Z,Us***,***@company.com,****-****-****-9667,140.107.***.**
USR_442,fdbd7e96-13f6-4888-a393-e5c83b12935b,2026-09-02T06:59:15.946Z,purchase,1,PROD_41,2,d0dd3f88-fe15-4732-8a52-0f8f11d123a4,70.28,null,/Volumes/workspace/default/raw_data/clickstream_landing/batch_6_1788332355.json/part-00000-tid-7492778453526864519-931ec837-d246-46dc-9ddd-bd773233d7de-1065-1-c000.json,2026-09-02T06:59:21.866Z,Us***,***@company.com,****-****-****-7578,52.62.***.**
USR_833,8fe15d04-ed67-4223-9e71-1c3647f33eaf,2026-09-02T06:59:15.946Z,cart,7,PROD_33,1,a86ea1f5-8cfb-43bb-83b8-9b53007a3544,32.54,null,/Volumes/workspace/default/raw_data/clickstream_landing/batch_6_1788332355.json/part-00000-tid-7492778453526864519-931ec837-d246-46dc-9ddd-bd773233d7de-1065-1-c000.json,2026-09-02T06:59:21.866Z,Us***,***@company.com,****-****-****-5837,40.112.***.**
USR_414,476b1198-7f9f-4856-ad45-a702e9e03128,2026-09-02T06:59:15.946Z,view,8,PROD_4,4,19af5242-fda8-4cfa-8497-e37a3d64adce,7.37,null,/Volumes/workspace/default/raw_data/clickstream_landing/batch_6_1788332355.json/part-00000-tid-7492778453526864519-931ec837-d246-46dc-9ddd-bd773233d7de-1065-1-c000.json,2026-09-02T06:59:21.866Z,Us***,***@company.com,****-****-****-5761,160.172.***.**
USR_715,785bf1b5-ae72-4512-85d1-55b8009da12c,2026-09-02T06:59:15.946Z,purchase,5,PROD_21,2,734411cd-64af-4ab3-add4-40ea41cfe15f,104.42,null,/Volumes/workspace/default/raw_data/clickstream_landing/batch_6_1788332355.json/part-00000-tid-7492778453526864519-931ec837-d246-46dc-9ddd-bd773233d7de-1065-1-c000.json,2026-09-02T06:59:21.866Z,Us***,***@company.com,****-****-****-1556,93.49.***.**
USR_579,abf1b065-86fc-4f65-bb5e-96b62004946f,2026-09-02T06:59:15.946Z,purchase,3,PROD_17,3,ad65bfbf-f79c-42ca-a268-fd6621257420,23.68,null,/Volumes/workspace/default/raw_data/clickstream_landing/batch_6_1788332355.json/part-00000-tid-7492778453526864519-931ec837-d246-46dc-9ddd-bd773233d7de-1065-1-c000.json,2026-09-02T06:59:21.866Z,Us***,***@company.com,****-****-****-1850,178.210.***.**
USR_487,30cc4ee4-ea7d-4140-b416-26af42f8cf77,2026-09-02T06:59:15.946Z,cart,9,PROD_32,4,ef0d46f4-ae30-4943-b664-7e29cdff09be,25.18,null,/Volumes/workspace/default/raw_data/clickstream_landing/batch_6_1788332355.json/part-00000-tid-7492778453526864519-931ec837-d246-46dc-9ddd-bd773233d7de-1065-1-c000.json,2026-09-02T06:59:21.866Z,Us*

{'status': 'success',
 'records_processed': 1002,
 'total_records': 11006,
 'mode': 'incremental',
 'last_cdf_version': 3}